# Evaluation Analyzer — Metadata Query Agent

Turn the metadata-query-agent's batch-evaluation output into a ranked list of prompt fixes — in minutes.

**The Problem:** An AgentCore batch run over our ground-truth dataset produces hundreds of per-evaluator score/explanation records (`per_query_events`) across metrics like `SqlGrounded`, `FinalAnswerFaithfulness`, and `Builtin.GoalSuccessRate`. Reading every explanation to find what's systematically wrong with the agent doesn't scale.

**The agent is a deterministic graph, not a ReAct loop.** The metadata-query-agent resolves each question with a fixed Tier 2 Strands graph (topic routing → slice build + sufficiency judge → SQL generate/validate → grounded execution). There is **no single `SYSTEM_PROMPT`**. The only model-facing, editable prompts are `JUDGE_PROMPT` (Phase 3 slice-sufficiency judge, which emits the `SliceSufficiency` decision) and `EXECUTION_PROMPT` (Phase 5 bounded execution agent, whose only tool is `execute_sql_query`). So a "fix" here is always a surgical edit to one of those two named prompts — and findings premised on a ReAct tool-ordering trajectory (e.g. "forbid the SliceSufficiency tool", "call disambiguate_query_terms first") are misreads of the deterministic graph and are flagged as **not prompt-addressable**.

**The Solution:** This notebook reads the batch-eval JSON our pipeline writes to `data/eval/results/metadata_query_batch_eval_*.json`, filters the low-scoring records, uses a two-agent (Strands) analysis to group failures into patterns, and generates surgical fixes for the two editable graph-phase prompts in `agents/metadata_query_agent/query_prompts.py`. Those prompts are read directly from that module — no manual copy step. You review the suggested edits, apply them, redeploy the agent, and re-run the eval to confirm improvement.

## Where This Fits

This is the **analyze → improve** step of the evaluation loop: run the ground-truth eval (notebook `2_metadata_query_agent_ondemand_groundtruth_eval.ipynb`) → analyze the results here → update the agent's graph-phase prompts → redeploy → re-evaluate. Repeat until scores meet your bar.

<p align="center">
<img src="../assets/images/improvement_loop.svg" alt="Continuous improvement loop" width="700">
</p>

## How It Works

Built with [Strands Agents SDK](https://github.com/strands-agents/sdk-python):

- **Orchestrator** receives all low-scoring evaluations + the live graph-phase prompts (`JUDGE_PROMPT`, `EXECUTION_PROMPT`, extracted from `query_prompts.py`)
- Calls `analyze_batch()` for each batch → invokes the **Batch Analyzer** sub-agent
- **Batch Analyzer** reads the judge explanations, groups them into failure patterns, tags each with a fix surface (`JUDGE_PROMPT` / `EXECUTION_PROMPT` / `NOT_PROMPT` / `DATA_GAP`), and extracts evidence quotes
- Orchestrator aggregates patterns across batches, ranks the prompt-addressable ones by frequency × severity, and separates out the ones no prompt change can fix
- Generates a report with the top 3 prompt-addressable problems and copy-paste-ready updated prompt(s)

<p align="center">
<img src="../assets/images/architecture.svg" alt="Architecture diagram" width="700">
</p>

## Input Source

- **Metadata-query-agent batch eval** — `data/eval/results/metadata_query_batch_eval_*.json` (per-evaluator records under `per_query_events`, roll-ups under `aggregate_summaries`; metric field is `evaluator`). This is the primary input. The notebook reads `EVAL_FOLDER` (`data/eval/results/`) and globs only `GLOB_PATTERN` (`metadata_query_batch_eval_*.json`) so other agents' runs in the same folder (metadata-enrichment, ontology-gen) are ignored.
- Also accepts generic [Strands](https://github.com/strands-agents/strands-evals) / [AWS AgentCore](https://docs.aws.amazon.com/agentcore/) session-level eval files (`{evaluator_name, score, explanation, trace_id, session_id}`).
- **Heads-up:** a batch whose `status` is `FAILED` usually has empty `per_query_events` — the notebook will report 0 evaluations and stop with guidance to re-run the eval.

## Step 1: Configuration

In [1]:
# Configuration
#
# Tuned for the semantic-layer metadata-query-agent. The eval notebooks write
# their batch output to data/eval/results/; this analyzer reads that folder
# directly. GLOB_PATTERN restricts it to the metadata-query-agent's own batch
# files (metadata_query_batch_eval_*.json) so a metadata-enrichment or ontology-gen run
# sitting in the same folder is NOT mixed into a query-agent prompt analysis.
EVAL_FOLDER = "../data/eval/results/"
GLOB_PATTERN = "metadata_query_batch_eval_*.json"  # only the metadata-query-agent batch files
BATCH_SIZE = 20  # per_query_events files hold many short records; larger batches = fewer LLM calls
SCORE_THRESHOLD = 0.7  # records scoring below this are treated as failures to analyze

# DEPLOYED SHAPE — read this before trusting any "tool" the judge mentions:
# the metadata-query-agent is NOT a free-form ReAct tool loop. It resolves each
# question with a deterministic Tier 2 Strands GRAPH
# (agents/metadata_query_agent/tier2/workflow.py) whose phases are plain Python
# function calls. There is NO module-level SYSTEM_PROMPT. The only model-facing,
# prompt-addressable surfaces on that path are:
#   * JUDGE_PROMPT      - Phase 3 slice-sufficiency judge (emits the
#                         `SliceSufficiency` {sufficient, missing} decision)
#   * EXECUTION_PROMPT  - Phase 5 bounded execution agent (the ONLY real model
#                         tool call: execute_sql_query)
# (Phase 4 SQL generation uses a one-line inline prompt in main.py; Phases 1/2
# topic-routing, term disambiguation and the grounding gate are deterministic
# Python, not prompts.) We read these straight from the agent module so this
# notebook always analyzes the prompts that are actually deployed.
AGENT_PROMPTS_SOURCE = "../agents/metadata_query_agent/query_prompts.py"
AGENT_PROMPT_VARS = ["JUDGE_PROMPT", "EXECUTION_PROMPT"]  # the editable graph-phase prompts
# Repo convention (see agents/metadata_query_agent/query_prompts.py): the
# cross-region inference profile id. Override with a us. profile if your
# account isn't onboarded to the global. profile.
MODEL_ID = "global.anthropic.claude-sonnet-5"

# ---------------------------------------------------------------------------
# Attach what the agent ACTUALLY did to each low-scoring judge record.
#
# WHY: the batch file's `per_query_events` carry ONLY the LLM judge's prose
# explanation — they do NOT carry the schema slice the judge scored. Without the
# slice, "is this `missing[]` name actually present?" is unanswerable, and an
# analyzer asked to find prompt fixes will GUESS "the judge over-rejected a name
# that was really in the slice" — a hallucinated premise (it cannot see the
# slice to confirm it). The sibling `agent_runs` array DOES record what the agent
# did: the executed `agent_sql`, the final `answer`, the returned `rows`, and
# whether it `clarified`. Joining that onto each judge record lets the analyzer
# distinguish the two failure shapes that look identical in a bare judge score:
#   * DEGRADED / real gap     -> agent_sql == "" and the answer says the data
#     isn't available (no SQL emitted -> nothing to over-reject; gap is upstream:
#     empty table, undeclared join, or an unmodelled relationship).
#   * GENUINE over-rejection  -> would require the named table/column to be
#     demonstrably present; cannot prove that from this file alone, so it must
#     NOT be the default assumption.
ATTACH_AGENT_RUNS = True  # set False to fall back to judge-explanation-only analysis

# Layer data-availability status, RE-VERIFIED directly against live AWS
# (huthmac+demo Admin, us-east-1) on 2026-06-13 by running EVERY
# Expected_SQL_Query in data/eval/groundtruth_dataset.json in Athena against the
# s3tablescatalog/semantic-layer-dev-analytics-tables / `normalized` database.
#
# RESULT: all 13 GT queries SUCCEED and return the expected rows. The earlier
# "empty participant/payout tables" data gap (see the now-superseded
# docs/research/enrichment-gaps-2026-06-12.md) is CLOSED — Track A is live
# (life_participant=50, rider_participant=9, holding_payout=10, annuity_detail=20).
# So this ledger does not assert "these questions cannot be answered"; instead it
# tells the analyzer the OPPOSITE — the data is present, so a degraded run on
# those questions is a slice/enrichment-DISCOVERY issue worth surfacing, not a
# data gap to wave through. The analyzer still has no data knowledge of its own,
# so we hand it this verified summary. RE-VERIFY (re-run the GT SQL) and rewrite
# this whenever the layer/data changes; clear it (set to "") for a layer you
# haven't verified.
KNOWN_DATA_GAPS = """
- DATA GAP CLOSED (verified 2026-06-13 by running every GT Expected_SQL_Query in
  Athena): the participant/payout tables are POPULATED (life_participant=50,
  rider_participant=9, holding_payout=10, annuity_detail=20) and ALL 13 GT
  queries return their expected rows. gt-row-00 (insured=policyholder, via a
  `life_participant` self-join), gt-row-01 (rider participants+roles, roles
  derived from `coverage.coverage_type`), gt-row-03 / gt-row-07 (holding<->party
  through the `coverage` bridge), and gt-row-04 (payout via `annuity_detail` +
  `financial_activity`) ARE answerable. A degraded run on any of these is
  therefore NOT a data gap — treat it as a slice/enrichment-DISCOVERY issue (the
  answer path exists in the data but the KB slice builder may not surface it) and
  analyze it on its merits. Do NOT wave it through as "correct degradation."
- Narrow schema facts that remain TRUE but block NO GT question: `rider_participant`
  exposes only holding_id / participant_sk / issue_age (NO party_id or role
  column) — gt-row-01 derives roles from `coverage` instead; and
  `coverage_participant`, `holding_owner`, `holding.owner_party_id` DO NOT EXIST —
  gt-row-00 answers via a `life_participant` self-join instead. So a `missing[]`
  entry naming one of these absent names signals the agent FIXATING ON THE WRONG
  path, not an unanswerable question — that is a slice/enrichment finding, not a
  reason to pass the failure off as a data gap.
- Multi-turn "How many are there?" scenarios (mt-parties-clarify,
  mt-stable-options): the agent DOES clarify and then answers correctly ("15
  parties"). A 0.0 GoalSuccess here is a JUDGE_PROMPT issue, NOT an eval-harness
  truncation: the judge sees the FULL multi-turn session but mistakes the
  deterministic graph's intermediate intent-classification JSON span for the
  agent's answer. The custom GoalSuccess judge (which replaced the un-editable
  Builtin.GoalSuccessRate) is told to read only the "Final-answer record" span,
  so these should now score 1.0. If they still read 0.0, treat it as JUDGE_PROMPT.
"""

# Load the editable graph-phase prompts by exec-ing the agent module in an
# isolated namespace (rather than importing it), so the notebook has no
# dependency on the agent package being importable and picks up nothing but the
# prompt constants.
from pathlib import Path

prompt_path = Path(AGENT_PROMPTS_SOURCE)
if not prompt_path.exists():
    raise FileNotFoundError(
        f"Agent prompt module not found: {AGENT_PROMPTS_SOURCE}. "
        "Run this notebook from the repo's notebooks/ directory."
    )

# query_prompts.py is a plain module of string constants (no imports / side
# effects), so exec-ing it to read the prompts is safe and dependency-free.
_prompt_ns: dict = {}
exec(compile(prompt_path.read_text(), str(prompt_path), "exec"), _prompt_ns)

_missing = [v for v in AGENT_PROMPT_VARS if v not in _prompt_ns]
if _missing:
    raise KeyError(
        f"{_missing} not defined in {AGENT_PROMPTS_SOURCE}. "
        f"Found: {[k for k in _prompt_ns if not k.startswith('__')]}"
    )

# Combine the editable graph-phase prompts into one labelled block the analysis
# agents reason over and propose fixes against. There is no single SYSTEM_PROMPT
# to rewrite, so a fix is always an edit to one of these named prompts.
_PHASE_LABEL = {
    "JUDGE_PROMPT": "Phase 3 slice-sufficiency judge (emits SliceSufficiency JSON)",
    "EXECUTION_PROMPT": "Phase 5 bounded execution agent (runs execute_sql_query)",
}
AGENT_PROMPTS_TEXT = "\n\n".join(
    f"### {v} - {_PHASE_LABEL.get(v, v)}\n{_prompt_ns[v].strip()}"
    for v in AGENT_PROMPT_VARS
)

## Step 2: Setup

In [2]:
import os
import json
import re
import time
from pathlib import Path
from typing import List, Dict, Any, Optional
from statistics import mean, stdev
from collections import defaultdict

import boto3
from botocore.config import Config
from dotenv import load_dotenv
from IPython.display import display, Markdown

from strands import Agent, tool
from strands.models import BedrockModel

# --- AWS auth (mirror notebook 2's pattern) ---------------------------------
# Strands' BedrockModel otherwise builds a client from the ambient credential
# chain, which is whatever the Jupyter kernel inherited at startup — easily a
# stale/expired identity. Pin to the profile/region declared in .env and hand
# the session to BedrockModel (cell below) so both notebooks share one
# refreshable SSO identity.
#
# AWS_PROFILE and AWS_REGION are read from .env (loaded here into os.environ).
# Defaults keep the notebook runnable when .env omits them; an explicit .env
# value always wins because load_dotenv populates os.environ before we read it.
load_dotenv(dotenv_path='.env', override=False)
AWS_PROFILE = os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
AWS_REGION = os.environ.get('AWS_REGION') or os.environ.get('AWS_DEFAULT_REGION') or 'us-east-1'
# Keep both region env vars in sync so any boto3 client built from the ambient
# environment (not just the session below) targets the same region.
os.environ['AWS_REGION'] = AWS_REGION
os.environ['AWS_DEFAULT_REGION'] = AWS_REGION

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
region = session.region_name or AWS_REGION

# Verify credentials up-front so an expired SSO token fails HERE with a clear
# message, not deep inside the first orchestrator call.
_sts = session.client(
    'sts', region_name=region,
    config=Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4'),
)
_identity = _sts.get_caller_identity()
print("✓ AWS Credentials Verified:")
print(f"  Profile  : {AWS_PROFILE}")
print(f"  Account  : ...{_identity['Account'][-4:]}")
print(f"  User/Role: {_identity['Arn'].split('/')[-1]}")
print(f"  Region   : {region}")

✓ AWS Credentials Verified:
  Profile  : huthmac+demo
  Account  : ...4087
  User/Role: huthmac-Isengard
  Region   : us-east-1


In [3]:
def _strip_evaluator_suffix(name: str) -> str:
    """Normalize a custom-evaluator id to a stable display name.

    The AgentCore batch runner suffixes custom evaluators with a per-run hash
    (e.g. ``FinalAnswerFaithfulness_72e74dbd`` or
    ``SqlGrounded_72e74dbd-1K4MeAH022``). Stripping the hash keeps records for
    the SAME evaluator grouped together across runs, while builtin evaluators
    (``Builtin.GoalSuccessRate``) are returned unchanged.
    """
    if not isinstance(name, str) or name.startswith("Builtin."):
        return name
    # Cut at the first ``_<hex>`` segment the batch runner appends.
    return re.sub(r"_[0-9a-f]{6,}(-\w+)?$", "", name)


def extract_evaluations(
    data: Any, parent_metadata: Optional[Dict] = None
) -> List[Dict]:
    """Recursively extract evaluations from any JSON structure.

    Handles the shapes our metadata-query-agent pipeline produces, plus the
    generic Strands / AgentCore session shape:

    1. **Query-agent batch output** (``data/eval/results/metadata_query_batch_eval_*.json``):
       per-evaluator records live under ``per_query_events`` (and roll-ups under
       ``aggregate_summaries``) and name the metric field ``evaluator`` (NOT
       ``evaluator_name``), with ``scenario_id`` and ``question`` for context.
       This is the primary input for this notebook.
    2. **Session-level evals** (Strands / AgentCore): a flat list of
       ``{evaluator_name, score, explanation, trace_id, session_id}``.

    A record is anything carrying BOTH a score (``score``/``value``) and an
    ``explanation``. Everything else is descended into. The metric name is
    folded into ``metadata['evaluator_name']`` (de-hashed) regardless of which
    field carried it, so downstream grouping is uniform.

    NOTE: a record needs an ``explanation`` to be picked up. A batch whose
    ``status`` is ``FAILED`` typically has empty ``per_query_events`` /
    ``aggregate_summaries`` (the run produced no per-evaluator judgments), so
    this returns nothing for it — re-run the eval to get a populated file.
    """
    evaluations = []
    parent_metadata = parent_metadata or {}

    if isinstance(data, list):
        for item in data:
            evaluations.extend(extract_evaluations(item, parent_metadata))
    elif isinstance(data, dict):
        score_key = (
            "score" if "score" in data else ("value" if "value" in data else None)
        )

        if score_key and "explanation" in data:
            eval_entry = {
                "score": data[score_key],
                "explanation": data["explanation"],
                "metadata": {**parent_metadata},
            }
            # Normalize the metric name: our batch output uses ``evaluator``;
            # the session shape uses ``evaluator_name``. Fold both into
            # ``evaluator_name`` (de-hashed) so grouping is consistent.
            raw_evaluator = data.get("evaluator") or data.get("evaluator_name")
            if raw_evaluator:
                eval_entry["metadata"]["evaluator_name"] = _strip_evaluator_suffix(
                    raw_evaluator
                )
            # Carry through any other scalar context fields verbatim. Use
            # setdefault so an inherited scenario_id/question from the parent
            # container is not clobbered by a same-named leaf field.
            for key, val in data.items():
                if key not in ["score", "value", "explanation", "evaluator"]:
                    if isinstance(val, (str, int, float, bool)):
                        eval_entry["metadata"].setdefault(key, val)
            evaluations.append(eval_entry)
        else:
            # Propagate context identifiers down into nested records so each
            # leaf evaluation knows which scenario / session it belongs to.
            nested_metadata = {**parent_metadata}
            for key in [
                "session_id",
                "trace_id",
                "evaluator_name",
                "evaluator_id",
                "scenario_id",
                "question",
                "agent_id",
                "eval_id",
            ]:
                if key in data and isinstance(data[key], (str, int, float, bool)):
                    nested_metadata[key] = data[key]
            # ``per_query_events`` / ``aggregate_summaries`` are the
            # metadata-query-agent batch containers; the others cover the
            # generic Strands / AgentCore shapes.
            for key in [
                "per_query_events",
                "aggregate_summaries",
                "results",
                "evaluations",
                "data",
                "items",
            ]:
                if key in data:
                    evaluations.extend(extract_evaluations(data[key], nested_metadata))

    return evaluations


def latest_matching_file(folder_path: str, pattern: str) -> Optional[Path]:
    """Return the single most-recent file (by mtime) matching ``pattern``.

    Only the latest run is analyzed, so a stale earlier ``metadata_query_batch_eval_*.json``
    sitting in the folder can never dilute or contradict the current run's
    results. Returns ``None`` when nothing matches.
    """
    matching = list(Path(folder_path).glob(pattern))
    if not matching:
        return None
    return max(matching, key=lambda p: p.stat().st_mtime)


def load_evaluations(folder_path: str, pattern: str = "*.json") -> List[Dict]:
    """Load the LATEST JSON file matching ``pattern`` and extract its evaluations.

    ``pattern`` is set to ``metadata_query_batch_eval_*.json`` by the config cell so only
    the metadata-query-agent's own batch output is analyzed (a metadata-enrichment
    or ontology-gen run in the same folder is a different agent and a different
    prompt — it must not be folded in).

    Only the single most-recent matching file (by mtime) is read — see
    :func:`latest_matching_file` — so a stale earlier run cannot pollute the
    analysis. To analyze a specific older run, point ``EVAL_FOLDER`` at a folder
    holding just that file.
    """
    evaluations = []
    latest = latest_matching_file(folder_path, pattern)
    if latest is None:
        return evaluations
    try:
        with open(latest, "r") as f:
            data = json.load(f)
        evals = extract_evaluations(data, {"source_file": latest.name})
        evaluations.extend(evals)
        status = data.get("status") if isinstance(data, dict) else None
        note = " [status=FAILED → no events to analyze]" if (
            not evals and status == "FAILED"
        ) else ""
        print(f"  {latest.name}: {len(evals)} evaluations{note}  (latest run only)")
    except Exception as e:
        print(f"  {latest.name}: Error - {e}")

    return evaluations


def compute_statistics(evaluations: List[Dict], threshold: float) -> Dict:
    """Compute statistics from evaluations."""
    if not evaluations:
        return {"total": 0, "error": "No evaluations found"}

    scores = [e["score"] for e in evaluations if e["score"] is not None]

    by_evaluator = defaultdict(list)
    for e in evaluations:
        evaluator = e["metadata"].get(
            "evaluator_name", e["metadata"].get("label", "unknown")
        )
        if e["score"] is not None:
            by_evaluator[evaluator].append(e["score"])

    evaluator_stats = {}
    for evaluator, eval_scores in by_evaluator.items():
        evaluator_stats[evaluator] = {
            "count": len(eval_scores),
            "mean": round(mean(eval_scores), 3),
            "min": round(min(eval_scores), 3),
            "max": round(max(eval_scores), 3),
        }
        if len(eval_scores) > 1:
            evaluator_stats[evaluator]["stdev"] = round(stdev(eval_scores), 3)

    low_scoring = [
        e for e in evaluations if e["score"] is not None and e["score"] < threshold
    ]

    return {
        "total": len(evaluations),
        "valid_scores": len(scores),
        "mean_score": round(mean(scores), 3) if scores else None,
        "min_score": round(min(scores), 3) if scores else None,
        "max_score": round(max(scores), 3) if scores else None,
        "stdev": round(stdev(scores), 3) if len(scores) > 1 else None,
        "low_scoring_count": len(low_scoring),
        "low_scoring_pct": round(len(low_scoring) / len(scores) * 100, 1)
        if scores
        else 0,
        "by_evaluator": evaluator_stats,
        "threshold": threshold,
    }


def batch_evaluations(evaluations: List[Dict], batch_size: int) -> List[List[Dict]]:
    """Split evaluations into batches."""
    return [
        evaluations[i : i + batch_size] for i in range(0, len(evaluations), batch_size)
    ]

In [4]:
# ---------------------------------------------------------------------------
# Join the agent's ACTUAL behaviour (agent_runs) onto each judge record.
#
# The batch file stores, alongside the judge `per_query_events`, an `agent_runs`
# array recording what the agent did per scenario turn: the executed `agent_sql`,
# the final `answer`, the returned `rows`, and the `clarified` flag. The judge
# records do NOT carry the schema slice that was scored, so on their own they
# can't tell "the judge over-rejected a present name" apart from "the agent
# honestly degraded on a real upstream gap". Attaching the run evidence closes
# that gap: an empty `agent_sql` with a "data isn't available" answer is honest
# degradation (no SQL emitted -> nothing was over-rejected), not a JUDGE_PROMPT
# bug. See the ATTACH_AGENT_RUNS / KNOWN_DATA_GAPS notes in the config cell.

def _scenario_of_session(scenario_session: str) -> Optional[str]:
    """Recover the scenario_id from an `agent_runs` ``scenario_session`` value.

    ``scenario_session`` is ``"<scenario_id>-<uuid>"`` (e.g.
    ``"gt-row-01-70d1fdde-d958-4768-8f1f-558092af4d58"``). The scenario_id is
    everything before the trailing UUID. We strip the canonical 5-group UUID
    suffix; if no UUID is present we return the whole string unchanged.
    """
    if not isinstance(scenario_session, str):
        return None
    # A v4-style UUID is 8-4-4-4-12 hex groups. Remove a trailing "-<uuid>".
    return re.sub(
        r"-[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$",
        "",
        scenario_session,
    )


def index_agent_runs(raw_batch: Any) -> Dict[str, List[Dict]]:
    """Group a batch file's ``agent_runs`` entries by scenario_id.

    Returns ``{scenario_id: [run_turn, ...]}`` with each turn trimmed to the
    behavioural fields the analyzer needs (turn_idx, message, answer, agent_sql,
    clarified, row_count). Turns are ordered by ``turn_idx`` so a multi-turn
    scenario reads chronologically. Returns ``{}`` when the file carries no
    ``agent_runs`` (older batch files, or a different agent's output).
    """
    runs_by_scenario: Dict[str, List[Dict]] = defaultdict(list)
    if not isinstance(raw_batch, dict):
        return {}
    for run in raw_batch.get("agent_runs", []) or []:
        if not isinstance(run, dict):
            continue
        scenario_id = _scenario_of_session(run.get("scenario_session", ""))
        if not scenario_id:
            continue
        rows = run.get("rows") or []
        runs_by_scenario[scenario_id].append(
            {
                "turn_idx": run.get("turn_idx"),
                "message": run.get("message"),
                "answer": run.get("answer"),
                # The actual SQL the agent ran (or "" when the agent degraded
                # without executing — the single strongest signal of degradation).
                "agent_sql": run.get("agent_sql", ""),
                "clarified": run.get("clarified", False),
                "row_count": len(rows) if isinstance(rows, list) else None,
            }
        )
    for scenario_id in runs_by_scenario:
        runs_by_scenario[scenario_id].sort(
            key=lambda t: t["turn_idx"] if t["turn_idx"] is not None else 0
        )
    return dict(runs_by_scenario)


def attach_agent_runs(
    low_scoring: List[Dict], runs_by_scenario: Dict[str, List[Dict]]
) -> int:
    """Attach the matching ``agent_runs`` turns to each low-scoring record.

    Mutates each record in place, adding ``record['metadata']['agent_behaviour']``
    (the list of turns for that scenario) and a derived
    ``record['metadata']['emitted_sql']`` boolean — True iff the agent ran any
    non-empty SQL on that scenario. ``emitted_sql == False`` is the machine-
    checkable tell of degradation (the agent did not execute SQL), which
    rules out the "judge over-rejected a present name" story for that record.

    Returns the count of records that were matched to at least one run, so the
    caller can warn if the join found nothing (e.g. an older file with no
    ``agent_runs``).
    """
    matched = 0
    for record in low_scoring:
        scenario_id = record["metadata"].get("scenario_id")
        turns = runs_by_scenario.get(scenario_id) if scenario_id else None
        if not turns:
            continue
        record["metadata"]["agent_behaviour"] = turns
        record["metadata"]["emitted_sql"] = any(
            (t.get("agent_sql") or "").strip() for t in turns
        )
        matched += 1
    return matched


## Step 3: Load and Validate Data

In [5]:
# Validate configuration
if not AGENT_PROMPTS_TEXT.strip():
    raise ValueError(
        f"No graph-phase prompts loaded. Check {AGENT_PROMPT_VARS} in {AGENT_PROMPTS_SOURCE}"
    )

print(
    f"Loaded {len(AGENT_PROMPT_VARS)} graph-phase prompt(s) "
    f"({len(AGENT_PROMPTS_TEXT)} chars) from {AGENT_PROMPTS_SOURCE}: "
    f"{', '.join(AGENT_PROMPT_VARS)}"
)

eval_path = Path(EVAL_FOLDER)
if not eval_path.exists():
    raise ValueError(f"Evaluation folder not found: {EVAL_FOLDER}")

# Analyze ONLY the latest matching batch file (by mtime) - not every JSON in the
# folder. Older metadata_query_batch_eval_*.json runs sitting alongside it would otherwise
# dilute / contradict the current run's results.
latest_file = latest_matching_file(EVAL_FOLDER, GLOB_PATTERN)
if latest_file is None:
    raise ValueError(
        f"No files matching {GLOB_PATTERN!r} in {EVAL_FOLDER}. "
        "Run notebook 2 (metadata_query_agent ondemand ground-truth eval) first, "
        "or adjust GLOB_PATTERN."
    )
all_matching = list(eval_path.glob(GLOB_PATTERN))
print(
    f"Found {len(all_matching)} file(s) matching {GLOB_PATTERN!r} in {EVAL_FOLDER}; "
    f"analyzing only the latest: {latest_file.name}"
)

Loaded 2 graph-phase prompt(s) (12460 chars) from ../agents/metadata_query_agent/query_prompts.py: JUDGE_PROMPT, EXECUTION_PROMPT
Found 1 file(s) matching 'metadata_query_batch_eval_*.json' in ../data/eval/results/; analyzing only the latest: metadata_query_batch_eval_20260625_194131.json


In [6]:
print(f"Loading evaluations from {EVAL_FOLDER} (pattern: {GLOB_PATTERN}):")
evaluations = load_evaluations(EVAL_FOLDER, GLOB_PATTERN)
print(f"\nTotal: {len(evaluations)} evaluations")

Loading evaluations from ../data/eval/results/ (pattern: metadata_query_batch_eval_*.json):
  metadata_query_batch_eval_20260625_194131.json: 48 evaluations  (latest run only)

Total: 48 evaluations


In [7]:
stats = compute_statistics(evaluations, SCORE_THRESHOLD)

# Guard the empty case: with no extracted records (e.g. every matching file was a
# FAILED batch with empty per_query_events), compute_statistics returns only
# {'total': 0, 'error': ...}. Stop here with a clear message rather than letting
# the score-formatting below raise KeyError, and skip the LLM analysis cells.
if stats["total"] == 0:
    low_scoring = []
    raise SystemExit(
        "No evaluations were extracted from the matching files.\n"
        "Most likely the batch run FAILED and produced empty per_query_events / "
        "aggregate_summaries (see the per-file notes above). Re-run notebook 2 to "
        "produce a SUCCEEDED metadata_query_batch_eval_*.json, then re-run this analyzer."
    )

print(
    f"Mean score: {stats['mean_score']} (range: {stats['min_score']} - {stats['max_score']})"
)
print(
    f"Low scoring (<{SCORE_THRESHOLD}): {stats['low_scoring_count']} ({stats['low_scoring_pct']}%)"
)

print("\nBy evaluator:")
for evaluator, eval_stats in stats["by_evaluator"].items():
    print(f"  {evaluator}: mean={eval_stats['mean']}, count={eval_stats['count']}")

low_scoring = [
    e for e in evaluations if e["score"] is not None and e["score"] < SCORE_THRESHOLD
]
print(f"\n{len(low_scoring)} low-scoring evaluations will be analyzed")

if not low_scoring:
    raise SystemExit(
        f"No records scored below {SCORE_THRESHOLD} — nothing to analyze. "
        "The agent passed every evaluator on this sample, or lower SCORE_THRESHOLD "
        "to inspect borderline cases."
    )

# Attach what the agent ACTUALLY did (executed SQL / final answer / clarified)
# to each low-scoring judge record, joined by scenario_id from the same batch
# file's `agent_runs`. This is the evidence that lets the analyzer tell honest
# degradation (no SQL emitted on a real upstream gap) apart from a genuine
# JUDGE_PROMPT over-rejection — without it the analyzer can only GUESS, and the
# guess is biased toward inventing a prompt fix. See the config-cell rationale.
if ATTACH_AGENT_RUNS:
    with open(latest_file, "r") as _f:
        _raw_batch = json.load(_f)
    _runs_by_scenario = index_agent_runs(_raw_batch)
    _matched = attach_agent_runs(low_scoring, _runs_by_scenario)
    if _runs_by_scenario:
        _degraded = sum(
            1 for e in low_scoring if e["metadata"].get("emitted_sql") is False
        )
        print(
            f"\nAttached agent_runs to {_matched}/{len(low_scoring)} low-scoring "
            f"records ({len(_runs_by_scenario)} scenarios in agent_runs)."
        )
        print(
            f"  {_degraded} of the matched records emitted NO SQL (honest "
            f"degradation — not a judge over-rejection)."
        )
    else:
        print(
            "\n[warn] ATTACH_AGENT_RUNS is on but this batch file has no "
            "`agent_runs` array — analyzing judge explanations only. The analyzer "
            "cannot verify whether a `missing[]` name was truly absent; it will "
            "default away from claiming JUDGE_PROMPT over-rejection."
        )


Mean score: 0.875 (range: 0.0 - 1.0)
Low scoring (<0.7): 6 (12.5%)

By evaluator:
  FinalAnswerFaithfulness: mean=0.812, count=16
  GoalSuccess: mean=0.812, count=16
  SqlGrounded: mean=1.0, count=16

6 low-scoring evaluations will be analyzed

Attached agent_runs to 6/6 low-scoring records (16 scenarios in agent_runs).
  4 of the matched records emitted NO SQL (honest degradation — not a judge over-rejection).


## Step 4: Define Analysis Agents

This step creates two agents using [Strands Agents SDK](https://github.com/strands-agents/sdk-python), both grounded in the metadata-query-agent's tools and evaluator semantics:

**Orchestrator** (main agent)<br>
**Batch Analyzer** (sub-agent)

**Flow:**  `Orchestrator` → `analyze_batch()` tool → `Batch Analyzer` → JSON patterns → Orchestrator synthesizes → `Report` → you update `agents/metadata_query_agent/query_prompts.py` and redeploy

In [8]:
# ============================================
# AGENT PROMPTS
# ============================================
#
# Grounded for the semantic-layer metadata-query-agent: a SQL-answering agent
# that resolves each question with a DETERMINISTIC Tier 2 Strands graph (topic
# routing -> slice build + sufficiency judge -> SQL generate/validate ->
# grounded execution), NOT a free-form ReAct tool loop. The only model-facing,
# editable prompts on that path are JUDGE_PROMPT (Phase 3) and EXECUTION_PROMPT
# (Phase 5), so every proposed fix names one of those two.

# Domain context shared by both agents so they reason about the RIGHT failure
# modes (bad slice-sufficiency calls, ungrounded SQL, execution-agent SQL
# rewrites) and do NOT hallucinate a ReAct tool-ordering story.
AGENT_DOMAIN_CONTEXT = (
    """
## Agent under evaluation — deterministic graph, NOT a ReAct tool loop
The metadata-query-agent resolves each question with a DETERMINISTIC Tier 2
Strands GRAPH (agents/metadata_query_agent/tier2/workflow.py). Phases are plain
Python function calls executed in a fixed order; the model does NOT choose tools:

  Phase 1  topic router          (KB retrieval -> candidate tables)                        [deterministic]
  Phase 2  term disambiguation   (ambiguous term -> clarification)                         [deterministic]
  Phase 3  slice builder + JUDGE (assemble schema slice; judge emits
                                  `SliceSufficiency` {sufficient, missing})                [JUDGE_PROMPT]
  Phase 3b slice disambiguation  (slice-level collisions -> clarification)                 [deterministic]
  Phase 4  SQL generate+validate (sqlglot parse + 1 repair; one-line inline prompt)        [inline prompt]
  Phase 5  grounding gate + bounded EXECUTION agent (only real model tool
                                  call: execute_sql_query)                                 [EXECUTION_PROMPT]

The only two prompts you can edit are JUDGE_PROMPT (Phase 3) and EXECUTION_PROMPT
(Phase 5). Every proposed fix must name exactly one of these. If a failure can't
be addressed by editing one of them, classify it NOT_PROMPT or DATA_GAP.

`SliceSufficiency` is the Phase 3 judge's expected structured output — not a
ghost tool. `retrieve_kb_context` and `disambiguate_query_terms` are deterministic
Python phases, not model tool calls — any judge framing that says the agent
"skipped" or "called them in the wrong order" is a misread of the fixed graph and
is NOT prompt-addressable.

## Evidence available — and the one trap to avoid
Each record gives you the LLM judge's `explanation` plus (when available) an
`agent_behaviour` block with each turn's `agent_sql`, `answer`, `clarified`, and
`row_count`, and a derived `emitted_sql` boolean.

**You do NOT have the schema slice the judge scored.** Therefore:
- If `emitted_sql == false` the agent degraded without executing SQL — nothing was
  over-rejected. Classify as NOT_PROMPT or DATA_GAP, never JUDGE_PROMPT.
- If `emitted_sql == true` but `SqlGrounded` fails, the judge PASSED a bad slice —
  that is a real JUDGE_PROMPT candidate.
- Do NOT assert "the judge over-rejected a name that was actually in the slice"
  unless the judge's own explanation states the name was present and was flagged
  anyway. Without that, the claim is unverifiable.

## Layer data-availability status for THIS layer (verified against live AWS — treat as ground truth)
The following was verified by running every ground-truth Expected_SQL_Query in
Athena. Read it BEFORE classifying any failure: it tells you which questions are
genuinely answerable from the data (so a degraded run is a slice/enrichment
finding to surface, NOT a data gap to excuse) and which residual facts are real
but block no GT question:
"""
    + KNOWN_DATA_GAPS
    + """
## What the evaluators mean
- GoalSuccess              — did the agent fully achieve the user's goal end-to-end (custom SESSION judge; replaced Builtin.GoalSuccessRate)?
- FinalAnswerFaithfulness  — does the final NL answer match the ground-truth expected answer?
- SqlGrounded              — does every table/column in the executed SQL appear in the
                             retrieved schema slice? (1.0 = no hallucinated schema; also
                             passes vacuously when the agent degraded and ran no SQL)

## Typical PROMPT-ADDRESSABLE failure modes (the only ones you should propose fixes for)
- JUDGE_PROMPT (Phase 3): the slice judge marks a slice sufficient when a needed
  table/column is missing (-> downstream SqlGrounded / Correctness fail), or marks
  it insufficient even though its OWN explanation shows the join path / column was
  present (-> needless clarification). The second only counts when the evidence
  shows the name was present — see the over-rejection trap above.
- EXECUTION_PROMPT (Phase 5): the execution agent rewrites/extends the grounded
  SQL instead of running it as-is, mishandles a zero-row result, retries on a
  non-transient error, or fails to surface the actual numeric result in its prose.
- Note: a low FinalAnswerFaithfulness score whose explanation cites a "PLACEHOLDER"
  expected answer is a GROUND-TRUTH DATA gap, not a prompt bug — flag it as such
  and do NOT propose a prompt change for it.
"""
)

BATCH_ANALYZER_PROMPT = (
    """
You analyze low-scoring evaluations of a SQL-generating data-warehouse agent to
identify systematic failure patterns. The agent is a DETERMINISTIC graph, not a
ReAct tool loop (see the agent description below) — keep that front of mind.
"""
    + AGENT_DOMAIN_CONTEXT
    + """
## Input
A batch of evaluations, each with:
- score: numeric 0-1 (scores < 0.7 indicate problems)
- explanation: detailed text from the LLM judge explaining why the score was given
- metadata: context including evaluator_name, scenario_id, question, session_id,
  and (when available) agent_behaviour [the turns the agent ran, each with
  agent_sql / answer / clarified / row_count] plus the derived emitted_sql flag

## Your Task
1. Read each explanation carefully - these contain the LLM judge's reasoning
2. CHECK metadata.emitted_sql / agent_behaviour before assigning a fix surface:
   if emitted_sql is false the agent ran no SQL (degraded) -> this is
   NOT_PROMPT or DATA_GAP, never JUDGE_PROMPT.
3. Identify SYSTEMATIC PATTERNS (not isolated incidents) in why scores are low
4. Group similar failures together (map each to the evaluator semantics above)
5. Extract 2-3 specific quotes as evidence per pattern
6. Note which evaluator metrics are affected and the scenario_id/question involved
7. For each pattern decide the fix surface: JUDGE_PROMPT, EXECUTION_PROMPT,
   NOT_PROMPT (needs a code change in a deterministic phase, the judge is
   mis-reading the deterministic graph as a ReAct trajectory, a known upstream
   data/enrichment/schema gap, or the agent honestly degraded), or DATA_GAP
   (PLACEHOLDER / unanswerable ground truth).

## Output (JSON)
Return ONLY valid JSON with this structure:
{
  "patterns": [
    {
      "name": "short descriptive name",
      "description": "what went wrong and why it's problematic",
      "count": N,
      "evaluators_affected": ["SqlGrounded", "GoalSuccess"],
      "evidence": ["direct quote from explanation 1", "direct quote from explanation 2"],
      "fix_surface": "JUDGE_PROMPT | EXECUTION_PROMPT | NOT_PROMPT | DATA_GAP",
      "root_cause": "what's missing/unclear in the named prompt, OR why it isn't prompt-addressable",
      "emitted_sql": true,
      "is_data_gap": false
    }
  ]
}

CONSTRAINTS:
- Maximum 5 patterns per batch
- Only include patterns that appear 2+ times (systematic, not isolated)
- Evidence quotes should be verbatim from explanations
- Do NOT propose forbidding `SliceSufficiency` or adding a `disambiguate_query_terms`
  tool — SliceSufficiency is the Phase 3 judge's expected output and the model
  has no such tools to call. Set fix_surface=NOT_PROMPT for any pattern whose
  premise is a ReAct trajectory this deterministic graph does not have.
- Do NOT tag a pattern JUDGE_PROMPT on the theory that a `missing[]` name was
  "really in the slice" — you cannot see the slice. Use NOT_PROMPT/DATA_GAP unless
  the judge's own explanation states the name was present and was flagged anyway.
- It is a CORRECT and useful outcome to find ZERO JUDGE_PROMPT/EXECUTION_PROMPT
  patterns when every failure is an upstream gap or degradation. Do not
  manufacture a prompt pattern to fill the list.
"""
)

ORCHESTRATOR_PROMPT = (
    """
You synthesize evaluation analysis of the metadata-query-agent into actionable
recommendations. Fixes are edits to one of the two editable graph-phase prompts
(JUDGE_PROMPT or EXECUTION_PROMPT) — there is NO single SYSTEM_PROMPT to rewrite.
"""
    + AGENT_DOMAIN_CONTEXT
    + """
## Your Task
1. Use the analyze_batch tool to analyze low-scoring evaluations in batches
2. Collect patterns from all batches
3. Identify the PROMPT-ADDRESSABLE problems (AT MOST 3, possibly FEWER, possibly
   ZERO) based on:
   - **Frequency**: Appears across multiple evaluations (the more, the worse)
   - **Severity**: Lower scores indicate more severe problems
   - **Fixability**: Can be fixed by editing JUDGE_PROMPT or EXECUTION_PROMPT, AND
     is supported by evidence (the agent emitted ungrounded SQL because a slice
     was wrongly passed, or the judge's own words show a present name flagged
     missing). Do NOT include a problem whose only basis is an unverifiable
     "the missing name was probably in the slice" guess.
4. Generate specific, minimal edits to the NAMED prompt for each REAL problem
5. EXCLUDE from the Top problems any pattern whose fix_surface is NOT_PROMPT or
   DATA_GAP (deterministic-phase code changes, ReAct-trajectory misreads, known
   upstream data/enrichment/schema gaps, degradation, or PLACEHOLDER
   ground truth) — list those separately under "Not prompt-addressable" so no one
   wastes time editing a prompt that can't fix them.
6. If NO pattern is genuinely prompt-addressable, say so plainly: write
   "No prompt-addressable problems found" under the Top Problems heading, leave
   the Suggested Prompt Changes table empty, and put everything under "Not
   prompt-addressable". This is a valid, valuable result — the score ceiling is
   then set upstream (data/enrichment/eval-harness), not by these two prompts.

## Required Output Format

# Evaluation Analysis Report

## Summary
[2-3 sentences on overall health of the agent based on the statistics and patterns
found. Call out the weakest evaluators by name. State how many low scores are NOT
prompt-addressable (degradation on a known upstream gap, ReAct-trajectory
misreads of the deterministic graph, or PLACEHOLDER ground-truth gaps) so the
failure rate isn't misread. If SqlGrounded is at/near 1.0, say so — it means the
agent is not hallucinating schema and a low GoalSuccess is likely upstream.
Use the per-evaluator means in the Evaluator Breakdown VERBATIM — do not restate
or re-round them.]

## Top Problems (prompt-addressable only — may be 0, 1, 2, or 3)

### Problem 1: [Specific Descriptive Name]

**Fix surface:** JUDGE_PROMPT or EXECUTION_PROMPT

**Evidence from evaluations:**
- "[Direct quote from LLM judge explanation]" (scenario_id, session_id)
- "[Another direct quote showing this pattern]" (scenario_id, session_id)
- [Cite agent_behaviour: did the agent emit SQL? what did it answer? This is how
  you prove the problem is real and prompt-addressable, not degradation.]

**Frequency & Impact:**
- Appears in X out of Y low-scoring evaluations
- Affects metrics: [list evaluator names]
- Average score when this occurs: X.XX

**Root Cause:**
[What's missing or unclear in the named graph-phase prompt that causes this]

**Proposed Fix:**
[Specific text to add/modify in JUDGE_PROMPT or EXECUTION_PROMPT and why it works]

---

(Problems 2 and 3 follow the same structure, each naming its fix surface. Include
only as many as the evidence genuinely supports — do NOT pad to three.)

---

## Not prompt-addressable
[Bulleted list of patterns with fix_surface NOT_PROMPT or DATA_GAP, each with a
one-line reason (deterministic-phase code change needed / judge misread the
deterministic graph as a ReAct loop / known upstream data-enrichment-schema gap /
degradation / PLACEHOLDER ground truth). No prompt edit is proposed for
these. For each, name the upstream owner of the fix if known (metadata-enrichment
agent, Glue MV schema, synthetic-data generator, eval harness).]

## Suggested Prompt Changes

### Changes Summary
| # | Prompt | What Changed | Original Text | New Text | Fixes |
|---|--------|--------------|---------------|----------|-------|
| 1 | JUDGE_PROMPT or EXECUTION_PROMPT | [brief] | [exact original snippet] | [exact new snippet] | Problem 1 |

(Leave this table empty — with a single row reading "No prompt-addressable
changes" — if you found no real prompt problem.)

### Complete Updated Prompts
For each prompt you changed, give the FULL updated string under its name so it can
be pasted straight into agents/metadata_query_agent/query_prompts.py:

#### JUDGE_PROMPT
```
[FULL UPDATED JUDGE_PROMPT — only if you changed it]
```

#### EXECUTION_PROMPT
```
[FULL UPDATED EXECUTION_PROMPT — only if you changed it]
```

## CONSTRAINTS
- At most 3 prompt-addressable problems, ranked by impact (frequency × severity);
  FEWER (including zero) is correct when the evidence doesn't support three
- Evidence must be actual quotes from the explanations with scenario_id and session_id
- Make minimal, surgical edits — do NOT rewrite a whole prompt, and do NOT
  duplicate a rule the prompt already contains (re-stating an existing rule louder
  does not fix a failure caused by an upstream data gap)
- NEVER invent a schema name (table, column, or FK) in a proposed fix or its
  rationale. Every table/column you reference must appear verbatim in a judge
  explanation, an agent_behaviour SQL string, or the prompt being edited. A fix
  that names a column the dataset may not have (e.g. an assumed `*.party_id` FK)
  is itself a hallucination — do not produce one.
- Every proposed change names exactly one prompt (JUDGE_PROMPT or EXECUTION_PROMPT)
- NEVER propose forbidding `SliceSufficiency` or adding a `disambiguate_query_terms`
  tool — both stem from misreading the deterministic graph as a ReAct loop
- Only output a "Complete Updated Prompt" for a prompt you actually changed
- No implementation roadmaps, KPIs, timelines, or risk assessments
"""
)


In [9]:
# ============================================
# INSTANTIATE THE ANALYSIS AGENTS
# ============================================
# Use the profile-bound session from the setup cell so this notebook
# authenticates the same way notebook 2 does (no ambient/expired credentials).
model = BedrockModel(model_id=MODEL_ID, boto_session=session)

batch_analyzer_agent = Agent(model=model, system_prompt=BATCH_ANALYZER_PROMPT)


@tool
def analyze_batch(batch_json: str) -> str:
    """Analyze a batch of low-scoring evaluations to identify failure patterns."""
    result = batch_analyzer_agent(
        f"Analyze these evaluations and return JSON patterns:\n{batch_json}"
    )
    return str(result)


orchestrator = Agent(
    model=model, system_prompt=ORCHESTRATOR_PROMPT, tools=[analyze_batch]
)

---

## Step 5: Run the Analysis

The orchestrator will:
1. Process evaluations in batches using the sub-agent
2. Identify the top 3 problems
3. Generate specific prompt improvements
4. Output updated JUDGE_PROMPT and/or EXECUTION_PROMPT snippets ready to paste

In [10]:
print(
    f"Analyzing {len(low_scoring)} evaluations in {len(batch_evaluations(low_scoring, BATCH_SIZE))} batches..."
)

batches = batch_evaluations(low_scoring, BATCH_SIZE)
batches_json = [json.dumps(batch, indent=2) for batch in batches]

analysis_prompt = f"""
Analyze these evaluation results and provide a comprehensive report.

## Statistics
- Total evaluations: {stats["total"]}
- Mean score: {stats["mean_score"]}
- Low scoring (<{SCORE_THRESHOLD}): {stats["low_scoring_count"]} ({stats["low_scoring_pct"]}%)
- Score range: {stats["min_score"]} - {stats["max_score"]}

## Evaluator Breakdown
{json.dumps(stats["by_evaluator"], indent=2)}

## Editable graph-phase prompts (the ONLY surfaces a fix may edit)
The agent has no single SYSTEM_PROMPT — it is a deterministic graph. The two
editable, model-facing prompts are below. Every proposed fix must edit one of
these by name (JUDGE_PROMPT or EXECUTION_PROMPT):

{AGENT_PROMPTS_TEXT}

## Low-Scoring Evaluations
There are {len(batches)} batches of evaluations to analyze.
Use the analyze_batch tool for each batch:

"""

for i, batch_json in enumerate(batches_json):
    analysis_prompt += f"\nBatch {i + 1}:\n{batch_json}\n"

start_time = time.time()
result = orchestrator(analysis_prompt)
elapsed_time = round(time.time() - start_time, 2)
result_text = str(result)

print(f"Analysis complete ({elapsed_time}s)")

Analyzing 6 evaluations in 1 batches...
I'll analyze the batch of low-scoring evaluations to identify failure patterns.
Tool #1: analyze_batch
```json
{
  "patterns": [
    {
      "name": "Slice judge marks sufficient for wrong join path — missing self-join and party-join columns cause spurious degradation",
      "description": "For gt-row-00 (insured=policyholder self-join via life_participant) and gt-row-04 (payout schedule via annuity_detail + financial_activity), the agent degrades without emitting any SQL, reporting columns like 'normalized.relation.holding_id', 'normalized.party', 'holding_payout.payout_mode', etc. as missing. Per the verified data-availability ground truth, both queries ARE answerable: gt-row-00 uses a life_participant self-join and gt-row-04 uses annuity_detail + financial_activity. The KB slice builder is surfacing the wrong candidate tables (or none at all), causing the Phase 3 judge to see a genuinely insufficient slice and correctly mark it missing — but 

## Step 6: View Results

After running this notebook:

- **Top 3 prompt-addressable problems** with evidence quotes from the LLM judge, each naming its fix surface (`JUDGE_PROMPT` or `EXECUTION_PROMPT`)
- **Root cause analysis** for each problem
- **Not prompt-addressable** section for patterns that stem from deterministic-phase code, ReAct-trajectory misreads, or PLACEHOLDER ground truth
- **Before/after changes table** plus the **complete updated prompt(s)** ready to paste into `agents/metadata_query_agent/query_prompts.py`

In [11]:
# ============================================
# DISPLAY RESULTS
# ============================================

display(Markdown(result_text))

Now I have all the information I need to produce the comprehensive report.

---

# Evaluation Analysis Report

## Summary

The agent is in reasonable overall health with a mean score of 0.875 across 48 evaluations, and — crucially — SqlGrounded is a perfect 1.0 (stdev 0.0), meaning the agent never hallucinates schema identifiers in the SQL it does emit. The weakest evaluators are FinalAnswerFaithfulness (mean: 0.812) and GoalSuccess (mean: 0.812), both showing identical distributions with a high stdev of 0.403 that reflects a small number of hard 0.0 failures dragging them down. Of the 6 low-scoring evaluations: 4 (gt-row-00 ×2, gt-row-04 ×2) are degradations caused by the KB/topic-router surfacing the wrong candidate tables — `emitted_sql == false` in all four, making them NOT prompt-addressable; 2 (gt-row-01 ×2) show the agent emitting SQL via the wrong join path because the Phase 3 judge passed a slice that lacked the `coverage` bridge needed to derive participant roles — these are genuinely JUDGE_PROMPT-addressable. No EXECUTION_PROMPT failures are present.

---

## Top Problems (prompt-addressable only — may be 0, 1, 2, or 3)

### Problem 1: Judge passes a slice for rider-role queries when the `coverage` bridge (carrying `coverage_type`) is absent, causing the SQL generator to use the wrong join path

**Fix surface:** JUDGE_PROMPT

**Evidence from evaluations:**
- "The assertion requires the final answer to match a specific expected result: participants identified via coverage records on the same policy holding, with roles from coverage type (Base, Optional, Rider), specific party IDs like PARTY000057, PARTY000100, PARTY000080…" (gt-row-01, GoalSuccess)
- "Each row links a rider (with its name, type, and status) to its rider participant and the associated insured life participant's party details — including full name, gender, underwriting class, tobacco basis, and issue gender — with riders ranging across types such as Child Term, Guaranteed Insurability, Critical Illness…" (gt-row-01, FinalAnswerFaithfulness)

The agent **did** emit SQL (`emitted_sql == true`, `row_count == 52`) and successfully executed it — confirming the judge passed the slice — but the executed SQL joins `rider` → `rider_participant` → `life_participant` → `party` and contains **no reference to `coverage` or `coverage_type`** anywhere. The ground-truth path requires joining through `coverage` to derive roles (`Base`, `Optional`, `Rider`) from `coverage_type`. The judge therefore passed a slice that was missing the `coverage` bridge, allowing SQL generation to proceed on the wrong path. SqlGrounded passes (no hallucination) but FinalAnswerFaithfulness and GoalSuccess both score 0.0.

**Frequency & Impact:**
- Appears in 2 out of 6 low-scoring evaluations (both dimensions of scenario gt-row-01)
- Affects metrics: FinalAnswerFaithfulness, GoalSuccess
- Average score when this occurs: 0.0

**Root Cause:**
The JUDGE_PROMPT's "Derivation & substitution" and "Relationship connectivity" rules currently allow the judge to see `rider_participant` in the slice and infer that roles are derivable — because the prompt says a "type/category/status column on a related table, reached over a present join, supplies it." In practice `rider_participant` has **no role or coverage_type column** (confirmed by the ground-truth note: `rider_participant` exposes only `holding_id / participant_sk / issue_age`). The judge is therefore being too permissive: it accepts `rider_type` (on the `rider` table itself) or `rider_participant` as a sufficient role source when the question specifically asks for participant roles — which require the `coverage` bridge. The prompt needs an explicit guard: when a question asks for the *roles* of *participants* (as distinct from the type of the rider product itself), the slice must contain a table with a column that discriminates participant role, reachable via a join that connects participants to that role column.

**Proposed Fix:**
Add a targeted clause to the JUDGE_PROMPT's "Derivation & substitution" section, immediately after the existing paragraph beginning "A role/category the question asks for is carried by a different but semantically-equivalent column on a JOINED row…":

> **Participant-role guard:** When the question asks for the *roles of participants* (e.g. "what role does each participant play", "insured participants and their roles"), verify the slice contains a column whose values identify participant roles (such as `coverage_type`) **on a table that joins to both the participant table and the policy/holding table**. A column describing the *product type* of the rider itself (e.g. `rider_type` or `rider_name`) is NOT a substitute for participant role — do not treat it as derivable. If no such participant-role column is present in the slice, set `sufficient=false` and name the missing table (the one carrying the participant-role/coverage-type column) in `missing[]`.

This is minimal and surgical: it does not change any other rule, and it directly prevents the judge from accepting a `rider`+`rider_participant` slice as sufficient for a participant-roles question when `coverage` (carrying `coverage_type`) is absent.

---

## Not prompt-addressable

- **gt-row-00 (×2 — FinalAnswerFaithfulness, GoalSuccess): Agent degrades, reports `normalized.relation.holding_id` and `normalized.relation.policy_id` as missing.** `emitted_sql == false`. The question is answerable via a `life_participant` self-join (verified in Athena). The Phase 1 topic router / KB retrieval is not surfacing `life_participant` as a candidate; instead it appears to retrieve a non-existent `relation` table. The judge then correctly marks that slice insufficient. Fix owner: **metadata-enrichment agent** — the KB entry for `life_participant` needs to describe the self-join pattern for "insured = policyholder" so the topic router retrieves it. No JUDGE_PROMPT or EXECUTION_PROMPT edit can compensate for the wrong tables being retrieved.

- **gt-row-04 (×2 — FinalAnswerFaithfulness, GoalSuccess): Agent degrades, reports `normalized.party`, `normalized.holding_payout.payout_mode`, `normalized.holding_payout.payout_amount`, `normalized.annuity_detail.payout_frequency`, `normalized.annuity_detail.payout_amount` as missing.** `emitted_sql == false`. The question is answerable via `annuity_detail` + `financial_activity` (verified in Athena). The KB slice builder is surfacing column names (e.g. `payout_mode`, `payout_amount` on `holding_payout`) that use different names than those actually present, and `normalized.party` is listed as missing even though it is a real table — suggesting the topic router is retrieving either the wrong tables or an incomplete slice. Fix owner: **metadata-enrichment agent** — KB entries for `annuity_detail`, `financial_activity`, and `holding_payout` need accurate column names and descriptions of the payout-path join so the router retrieves the correct slice. Again, `emitted_sql == false` means no JUDGE_PROMPT or EXECUTION_PROMPT edit applies.

---

## Suggested Prompt Changes

### Changes Summary

| # | Prompt | What Changed | Original Text | New Text | Fixes |
|---|--------|--------------|---------------|----------|-------|
| 1 | JUDGE_PROMPT | Add participant-role guard clause after the "A role/category the question asks for…" derivation sentence | `A role/category the question asks for is carried by a different but semantically-equivalent column on a JOINED row (a type/category/status column on a related table, reached over a present join, supplies it) — do NOT require a literally-named role column when such an equivalent column is present on a related table reachable in the slice.` | `A role/category the question asks for is carried by a different but semantically-equivalent column on a JOINED row (a type/category/status column on a related table, reached over a present join, supplies it) — do NOT require a literally-named role column when such an equivalent column is present on a related table reachable in the slice. **Participant-role guard:** When the question asks for the *roles of participants* (e.g. "what role does each participant play", "insured participants and their roles"), verify the slice contains a column whose values identify participant roles (such as `coverage_type`) on a table that joins to both the participant table and the policy/holding table. A column describing the *product type* of the rider itself (e.g. `rider_type`) is NOT a substitute for participant role — do not treat it as derivable from rider-product columns. If no such participant-role column is present in the slice, set `sufficient=false` and name the missing table (the one carrying the participant-role column) in `missing[]`.` | Problem 1 |

---

### Complete Updated Prompts

#### JUDGE_PROMPT

```
You decide whether a retrieved schema slice is sufficient to answer a user question. Output SliceSufficiency JSON only.

If the slice contains the tables and columns needed to write an Athena SQL query for the question, set sufficient=true and missing=[].
If tables or columns are missing, set sufficient=false and list the table/column names you'd want added to the slice.
Bias: prefer sufficient=true when the slice contains a clear join path between the tables the question references; only flag missing when a critical table or column is absent. If the question requires the same table in two roles (e.g. a self-join via a role/type column) or a single FK traversal with a status/type filter, mark sufficient=true as long as the relevant columns (role, status, FK) are present anywhere in the slice — do not flag them as missing merely because the question uses two semantic labels for the same column. Read a column's or table's DESCRIPTION as authoritative guidance — when it names the join key, the lookup, or the derivation to use, honor it rather than demanding a separate concept.
Column-level completeness (this OVERRIDES the join-path bias): before setting sufficient=true, break the question into the concrete pieces of data it asks for (each value to return, filter on, group by, or order by) and confirm EACH maps to an actual column present in the slice. A clear join path is NOT enough on its own — but a requested value counts as PRESENT if a column carries it DIRECTLY *or* it can be DERIVED/SUBSTITUTED from columns that are present (see 'Derivation & substitution' below). Only after applying that derivation test — if a requested value has neither a direct backing column NOR a derivable/substitutable source in the slice — set sufficient=false and name the genuinely-absent column in missing[]. When the question asks for a 'human-readable description', 'label', 'name', or 'type' of a value, the slice satisfies it in EITHER of two ways — accept the FIRST that applies and set sufficient=true:
  (a) A column already carrying the readable value is present. A column tagged "semantic_role": "label" in the slice IS the human-readable form of its coded sibling (its values are themselves descriptive words, so it needs no further lookup). Likewise a column whose values are inherently readable (a name, a type word) needs no further lookup. Do NOT demand a separate lookup table when such a label/value column is present — that is over-rejection. When path (a) applies (a label or inherently-readable column is present), confirm sufficient=true on that basis ALONE and do NOT put any lookup/reference table in missing[] or treat its absence as a gap — the readable value is already available in a direct column, so the SQL must read it directly and must NOT introduce a JOIN to a code/lookup table (an INNER JOIN to a lookup whose codes don't line up silently drops ALL rows → a wrong 0-row answer).
  (b) Only when NO readable label column exists for the coded value AND a separate lookup/reference table (a code→description table joined on the code) would be required, and that lookup table/column is absent from the slice — then set sufficient=false and name the missing lookup. Do NOT assume a convenient column exists; verify it by name (or by a "label" semantic_role) in the slice.
Table-name verification (also OVERRIDES the join-path bias): for every table the question would require in SQL, confirm a table with that name appears in the slice's `tables` array. Match on the TABLE name only — NEVER invent or require a particular schema/database qualifier from the question text. Any layer/database name the USER typed (e.g. words like 'curated', 'normalized', 'raw', or 'the … layer') is prose describing the dataset, NOT part of a required table id; do not treat it as a qualifier the slice must match. If the needed table name is present in `tables` (under whatever schema the slice uses), it is available — set sufficient=true on that basis. Only set sufficient=false and name the missing table in missing[] when NO table of that name exists in the slice — not merely because its schema prefix differs from a word in the question. Do NOT assume a table exists under an unrelated name, and do NOT fabricate a schema-qualified id (a `<schema>.<table>` string you constructed) that does not appear in the slice.
Self-check before emitting sufficient=false: for EVERY name you are about to add to missing[], strip any schema/catalog prefix (everything up to and including the last dot) and re-check whether that bare table name appears in the slice's `tables` array, or that bare column name appears in `columns`. If the bare name IS present, do NOT add it — a schema prefix you constructed is not a required qualifier. Only keep an entry in missing[] when this strip-and-recheck still finds no match.
Entity-table disambiguation: if the question asks for a count or list of a named business entity and the slice contains both a PRIMARY entity table AND a derived/associative table for that entity (a table whose name extends the entity's name with a role/link suffix), set sufficient=false and name the primary entity table in missing[] unless the primary table is explicitly present in the slice. Do not treat a derived or associative table as a substitute for counting the primary entity.
Relationship connectivity (also OVERRIDES the join-path bias): when the question RELATES two entities (e.g. 'measure of X items grouped BY Y', 'X and their related Z'), verify the slice actually contains a join PATH connecting those two tables — either a direct foreign key, or a bridge / junction / intermediate table that joins to BOTH (two entity tables may not join directly and connect only through a third table that carries both their keys). A multi-hop path counts: if entity A joins to a bridge table and that bridge (directly or via another present table) carries an FK to entity B, the two are connected and the relationship IS satisfied — set sufficient=true on that basis. A bridge that joins to BOTH is fully equivalent to a direct foreign key: when such a bridge is present (a table holding both entities' keys), do NOT additionally demand a direct FK on an entity table, and do NOT invent a missing direct-FK column or require a generic 'relation' table — naming an absent direct-FK column or a redundant linking table when a connecting bridge already exists in the slice is over-rejection. Only set sufficient=false for connectivity when the two tables are present but NOTHING in the slice connects them — then name the bridge / junction table you'd add in missing[] (the builder will pull it in). If NO table in this dataset can connect them — the relationship the question needs simply is not modelled — set sufficient=false and name the missing linking table; do NOT pass a slice the SQL generator could only satisfy by inventing a join column that does not exist.
Do NOT FABRICATE a missing table name. Every entry you put in missing[] must be a plausibly-real table for THIS dataset, not an invented convenience name (do NOT emit a guessed composite name like '<entityA>_<entityB>' as missing — those are guesses, not real tables). Before naming a relationship as unsatisfied, check whether a table ALREADY in the slice carries the keys the relationship needs and can be SELF-JOINED or filtered to express it. In particular, an association table that holds BOTH a parent-record key AND a related-entity key (plus a row discriminator) lets you express 'the same related entity appearing in two roles on one parent' by a self-join / GROUP BY … HAVING on its own rows — no separate role/membership table is required. If such a table is present, the relationship IS satisfied → sufficient=true; reserve sufficient=false for when no slice table carries the needed keys at all.
Derivation & substitution before missing[] (this OVERRIDES the column-level completeness check): a question does NOT require a column literally named after its surface wording. Before adding any column to missing[], ask whether the requested value can be PRODUCED from columns already in the slice — by deriving it from a related row, or by substituting an equivalent source reached over a present join path. Questions are often deliberately answerable through enrichment joins rather than a single conveniently-named column, so reaching for a literal column and degrading is the dominant false negative. A value is DERIVABLE/SUBSTITUTABLE when:
  - A role/category the question asks for is carried by a different but semantically-equivalent column on a JOINED row (a type/category/status column on a related table, reached over a present join, supplies it) — do NOT require a literally-named role column when such an equivalent column is present on a related table reachable in the slice. Participant-role guard: When the question asks for the *roles of participants* (e.g. "what role does each participant play", "insured participants and their roles"), verify the slice contains a column whose values identify participant roles (such as coverage_type) on a table that joins to both the participant table and the policy/holding table. A column describing the *product type* of the rider itself (e.g. rider_type) is NOT a substitute for participant role — do not treat rider-product columns as derivable sources for participant role. If no such participant-role column is present in the slice, set sufficient=false and name the missing table (the one carrying the participant-role column) in missing[].
  - A measure the question asks for lives on a RELATED transaction/detail table reachable via a present join, even when the obviously-named table for it is empty or column-less. When an equivalent measure column on a related, joinable table IS present, the obviously-named table being empty or lacking the measure column is NOT a gap — use the equivalent source.
  - A relationship the question states is expressible by SELF-JOINING or GROUP BY…HAVING on a single association table that already holds the needed keys (one table carrying both a parent-record key and a related-entity key, plus a row discriminator) — do NOT require a literally-named owner/role column or a dedicated role/membership table.
Apply this test to EVERY candidate missing[] entry. Only keep an entry when NO present column carries the value directly AND none of the above derivation/substitution routes can produce it from the slice's columns and join paths. When a derivation or substitution route exists, set sufficient=true and leave missing[] empty for that value — the SQL generator is instructed to build the enrichment join, and the Phase 5 grounding gate remains the backstop against truly hallucinated identifiers.
Evidence source (read carefully): judge column availability ONLY from the slice's `columns` array — each entry is {table_id, name, type, ...}. A name that appears in the `tables` array but has NO matching {table_id, name} entry in `columns` is NOT present in the slice; do not treat it as available. In particular, never flip to sufficient=true because a column name appears as a dotted `table.column` string in `tables`. When a needed column is absent from `columns`, set sufficient=false and name it in missing[].
```


## Step 7: Save Results

In [12]:
import os
from datetime import datetime

# Write the report into data/eval/results-analysis/ (sibling of the EVAL_FOLDER
# the batch results live in), using a notebook-relative path so the output
# location is deterministic regardless of the process cwd — the bare filename
# would otherwise land wherever the kernel happened to be launched from.
ANALYSIS_FOLDER = "../data/eval/results-analysis/"
os.makedirs(ANALYSIS_FOLDER, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = os.path.join(ANALYSIS_FOLDER, f"analysis_report_{timestamp}.md")

report_content = f"""# Evaluation Analysis Report

**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
**Evaluations Analyzed:** {len(low_scoring)} low-scoring out of {stats["total"]} total
**Processing Time:** {elapsed_time}s

---

{result_text}
"""

with open(output_file, "w") as f:
    f.write(report_content)

print(f"Report saved to {output_file}")

Report saved to ../data/eval/results-analysis/analysis_report_20260625_230932.md


---

## Next Steps

After running this analysis:

1. **Review the report** — Check the top 3 prompt-addressable problems against the `aggregate_summaries` roll-up in the source file. Be skeptical of any finding that assumes a ReAct tool-ordering trajectory (e.g. "the agent skipped retrieve_kb_context" or "it called the SliceSufficiency tool instead of disambiguate_query_terms") — the deployed agent is a deterministic graph, so those are misreads and the report should list them under "Not prompt-addressable".
2. **Apply the prompt edits** — For each problem, paste the "Complete Updated Prompt" into the named string (`JUDGE_PROMPT` or `EXECUTION_PROMPT`) in `agents/metadata_query_agent/query_prompts.py`. On the next run, this notebook re-reads those same variables, so your edits are picked up automatically.
3. **Test incrementally** — Apply one change at a time if you want to isolate its effect.
4. **Redeploy** — `cdk deploy semantic-layer-dev-agentcore` rebuilds and ships the agent runtime.
5. **Re-evaluate** — Re-run `notebooks/2_metadata_query_agent_ondemand_groundtruth_eval.ipynb`; it writes a new `metadata_query_batch_eval_*.json` into `data/eval/results/` (the `EVAL_FOLDER`), then re-run this analyzer to measure improvement.
6. **Iterate** — Repeat until scores meet your target.

### Tips

- If a finding can't be fixed by editing `JUDGE_PROMPT` or `EXECUTION_PROMPT`, it isn't a prompt bug — it needs a code change in a deterministic phase (`tier2/`) or a ground-truth fix. Don't force a prompt edit.
- If you have many low-scoring records, raise `BATCH_SIZE` to reduce LLM calls.
- A `status=FAILED` batch with 0 events means the eval run itself failed (not that the agent is perfect) — re-run notebook 2 before analyzing.